# Sistema Integral de Gestión de Clientes, Servicios y Reservas — **Software FJ**

**Universidad Nacional Abierta y a Distancia – UNAD**
Curso: Programación (213023) · Fase 4 – Componente práctico – Prácticas simuladas

**Objetivo:** construir una aplicación orientada a objetos, **sin bases de datos**, estable, modular y extensible, que aplique
abstracción, herencia, polimorfismo, encapsulación y manejo avanzado de excepciones.

| Sección | Contenido | Responsable |
|---|---|---|
| 1 | Configuración del archivo de logs | Integrante 1 |
| 2 | Excepciones personalizadas | Integrante 1 |
| 3 | Clase abstracta `EntidadBase` y clase `Cliente` | Integrante 2 |
| 4 | Clase abstracta `Servicio` y servicios especializados | Integrante 3 |
| 5 | Clase `Reserva` | Integrante 4 |
| 6 | Clase `SistemaGestion` (listas internas) | Integrante 5 |
| 7 | Simulación de operaciones y revisión del log | Todo el equipo |

> Ejecute las celdas **en orden** (Kernel → Restart & Run All).

## 1. Configuración del archivo de logs
Todos los eventos y errores se guardan en `software_fj.log`. Es el **único** archivo que usa el sistema
(la información de clientes, servicios y reservas vive en objetos y listas).

In [1]:
# --- Importaciones ---------------------------------------------------------
import logging                                   # Módulo estándar para registrar eventos en archivos
import re                                        # Expresiones regulares para validar correo y teléfono
from abc import ABC, abstractmethod              # Herramientas para crear clases abstractas
from datetime import datetime                    # Fecha y hora de creación de entidades
from functools import singledispatchmethod      # Permite "sobrecargar" métodos según el tipo del argumento

ARCHIVO_LOG = "software_fj.log"                  # Nombre del archivo donde se registran eventos y errores

logger = logging.getLogger("SoftwareFJ")         # Se crea (u obtiene) un logger con nombre propio
logger.setLevel(logging.DEBUG)                   # Se registran todos los niveles: DEBUG, INFO, WARNING, ERROR
logger.handlers.clear()                          # Evita handlers duplicados si la celda se ejecuta varias veces

manejador = logging.FileHandler(ARCHIVO_LOG, mode="w", encoding="utf-8")  # "w": log limpio en cada ejecución
formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s",  # Formato: fecha | nivel | mensaje
                            datefmt="%Y-%m-%d %H:%M:%S")
manejador.setFormatter(formato)                  # Se asigna el formato al manejador de archivo
logger.addHandler(manejador)                     # Se conecta el manejador al logger

logger.info("===== Inicio del Sistema Software FJ =====")  # Primer evento registrado
print(f"Logger configurado. Los eventos se guardan en '{ARCHIVO_LOG}'.")

Logger configurado. Los eventos se guardan en 'software_fj.log'.


## 2. Excepciones personalizadas
Se crea una jerarquía propia: todas heredan de `SoftwareFJError`, así el sistema puede capturar
cualquier error del dominio con un solo `except` sin ocultar errores inesperados de Python.

In [2]:
class SoftwareFJError(Exception):
    """Excepción base del sistema. Todas las demás heredan de ella."""
    def __init__(self, mensaje, codigo="FJ-000"):
        super().__init__(mensaje)                # Guarda el mensaje en la excepción estándar
        self.codigo = codigo                     # Código interno para identificar el tipo de error

    def __str__(self):
        return f"[{self.codigo}] {self.args[0]}" # Muestra el código junto al mensaje


class ClienteInvalidoError(SoftwareFJError):
    """Datos personales del cliente inválidos o faltantes."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-100")


class ServicioInvalidoError(SoftwareFJError):
    """Servicio mal configurado (tarifa, capacidad, parámetros)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-200")


class ServicioNoDisponibleError(SoftwareFJError):
    """El servicio existe pero no está disponible para reservar."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-210")


class ReservaInvalidaError(SoftwareFJError):
    """Intento de reserva con datos incorrectos."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-300")


class OperacionNoPermitidaError(SoftwareFJError):
    """Operación prohibida según el estado actual (ej.: cancelar algo ya procesado)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-400")


class CalculoInconsistenteError(SoftwareFJError):
    """Un cálculo de costo dio un resultado ilógico (negativo, descuento > 100 %, etc.)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-500")


print("Excepciones personalizadas definidas:",
      [c.__name__ for c in SoftwareFJError.__subclasses__()])

Excepciones personalizadas definidas: ['ClienteInvalidoError', 'ServicioInvalidoError', 'ServicioNoDisponibleError', 'ReservaInvalidaError', 'OperacionNoPermitidaError', 'CalculoInconsistenteError']


## 3. Clase abstracta `EntidadBase` y clase `Cliente`
`EntidadBase` representa cualquier entidad general del sistema (cliente, servicio, reserva): le da un
identificador único y una fecha de creación, y **obliga** a las hijas a implementar `describir()` y `validar()`.

`Cliente` aplica **encapsulación**: los atributos son privados (`__nombre`, `__documento`…) y solo se
modifican mediante *properties* que validan cada dato.

In [3]:
class EntidadBase(ABC):
    """Clase abstracta que representa cualquier entidad general del sistema."""
    _contador = 0                                # Contador de clase para generar IDs únicos

    def __init__(self, prefijo):
        EntidadBase._contador += 1               # Incrementa el contador compartido
        self._id = f"{prefijo}-{EntidadBase._contador:03d}"  # Ej.: CLI-001, SRV-004, RES-010
        self._fecha_creacion = datetime.now()    # Momento en que se creó la entidad

    @property
    def id(self):
        return self._id                          # Solo lectura: el ID no se puede cambiar

    @abstractmethod
    def describir(self):
        """Cada entidad debe saber describirse (polimorfismo)."""

    @abstractmethod
    def validar(self):
        """Cada entidad debe validar su propio estado."""

    def __str__(self):
        return self.describir()                  # print(entidad) usa la descripción polimórfica


class Cliente(EntidadBase):
    """Cliente de Software FJ con datos personales encapsulados y validados."""
    PATRON_EMAIL = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.-]+$")  # Formato básico de correo

    def __init__(self, nombre, documento, email, telefono):
        super().__init__("CLI")                  # Genera el ID con prefijo CLI
        # Cada asignación pasa por su property, por lo tanto se valida automáticamente
        self.nombre = nombre
        self.documento = documento
        self.email = email
        self.telefono = telefono

    # ----- nombre -----
    @property
    def nombre(self):
        return self.__nombre                     # Atributo privado (name mangling)

    @nombre.setter
    def nombre(self, valor):
        if not isinstance(valor, str) or not valor.strip():          # Vacío o no es texto
            raise ClienteInvalidoError("El nombre es obligatorio.")
        if any(c.isdigit() for c in valor):                           # No se aceptan números
            raise ClienteInvalidoError(f"El nombre '{valor}' no puede contener números.")
        self.__nombre = valor.strip().title()                         # Se guarda normalizado

    # ----- documento -----
    @property
    def documento(self):
        return self.__documento

    @documento.setter
    def documento(self, valor):
        valor = str(valor).strip()                                    # Acepta int o str
        if not valor.isdigit() or not 6 <= len(valor) <= 10:          # Solo dígitos, 6 a 10
            raise ClienteInvalidoError(f"Documento '{valor}' inválido (6 a 10 dígitos).")
        self.__documento = valor

    # ----- email -----
    @property
    def email(self):
        return self.__email

    @email.setter
    def email(self, valor):
        if not isinstance(valor, str) or not self.PATRON_EMAIL.match(valor):
            raise ClienteInvalidoError(f"Correo '{valor}' con formato inválido.")
        self.__email = valor.lower()

    # ----- teléfono -----
    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        valor = str(valor).strip()
        if not re.fullmatch(r"3\d{9}", valor):                        # Celular colombiano: 10 dígitos, inicia en 3
            raise ClienteInvalidoError(f"Teléfono '{valor}' inválido (10 dígitos iniciando en 3).")
        self.__telefono = valor

    def validar(self):
        return True                              # Si el objeto existe, ya pasó todas las validaciones

    def describir(self):
        doc_oculto = "*" * (len(self.documento) - 3) + self.documento[-3:]  # Protege el dato personal
        return f"Cliente {self.id}: {self.nombre} | Doc {doc_oculto} | {self.email}"


# Prueba rápida de la celda
c_prueba = Cliente("ana pérez", "1012345678", "ana@correo.com", "3101234567")
print(c_prueba)

Cliente CLI-001: Ana Pérez | Doc *******678 | ana@correo.com


## 4. Clase abstracta `Servicio` y servicios especializados
Tres servicios heredan de `Servicio`: **ReservaSala**, **AlquilerEquipo** y **Asesoria**. Cada uno
**sobrescribe** `calcular_costo()`, `describir()` y `validar_parametros()` (polimorfismo).

**Sobrecarga de métodos.** Python no permite dos métodos con el mismo nombre, así que se usan las dos
formas propias del lenguaje:
1. `calcular_costo_total(horas, impuesto=..., descuento=...)` — parámetros opcionales (variantes con/sin impuesto y descuento).
2. `cotizar(...)` con `@singledispatchmethod` — se comporta distinto según el **tipo** del argumento (`int`, `float` o `dict`).

In [4]:
class Servicio(EntidadBase):
    """Clase abstracta para todos los servicios de Software FJ."""
    IVA = 0.19                                   # Impuesto por defecto (19 %)

    def __init__(self, nombre, tarifa_hora, disponible=True):
        super().__init__("SRV")
        if not nombre or not str(nombre).strip():
            raise ServicioInvalidoError("El servicio necesita un nombre.")
        if not isinstance(tarifa_hora, (int, float)) or tarifa_hora <= 0:
            raise ServicioInvalidoError(f"Tarifa '{tarifa_hora}' inválida: debe ser un número mayor que 0.")
        self._nombre = str(nombre).strip()       # Atributo protegido (uso interno y de las hijas)
        self._tarifa_hora = float(tarifa_hora)
        self._disponible = disponible

    @property
    def nombre(self):
        return self._nombre

    @property
    def disponible(self):
        return self._disponible

    def cambiar_disponibilidad(self, estado: bool):
        self._disponible = estado                # Permite habilitar/deshabilitar el servicio
        logger.info(f"{self.id} '{self.nombre}' disponible = {estado}")

    # ---- Métodos abstractos que cada hija DEBE sobrescribir ----
    @abstractmethod
    def calcular_costo(self, horas):
        """Costo base del servicio según sus reglas propias."""

    @abstractmethod
    def validar_parametros(self, horas):
        """Valida que la duración solicitada sea aceptable para este servicio."""

    def validar(self):
        return self._tarifa_hora > 0

    # ---- Sobrecarga 1: parámetros opcionales ----
    def calcular_costo_total(self, horas, impuesto=None, descuento=0.0):
        """
        Variantes:
          calcular_costo_total(h)                    -> con IVA por defecto
          calcular_costo_total(h, impuesto=0)        -> sin impuesto
          calcular_costo_total(h, descuento=0.10)    -> con IVA y 10 % de descuento
        """
        impuesto = self.IVA if impuesto is None else impuesto
        if not 0 <= descuento < 1:
            raise CalculoInconsistenteError(f"Descuento {descuento:.0%} fuera del rango permitido (0 % a 99 %).")
        if impuesto < 0:
            raise CalculoInconsistenteError("El impuesto no puede ser negativo.")
        base = self.calcular_costo(horas)                 # Llamada polimórfica: cada hija calcula distinto
        total = base * (1 - descuento) * (1 + impuesto)
        if total <= 0:
            raise CalculoInconsistenteError(f"Costo total inconsistente: {total}")
        return round(total, 2)

    # ---- Sobrecarga 2: según el tipo del argumento ----
    @singledispatchmethod
    def cotizar(self, arg):
        raise ServicioInvalidoError(f"No se puede cotizar con un argumento de tipo {type(arg).__name__}.")

    @cotizar.register
    def _(self, horas: int):                              # cotizar(3) -> horas enteras
        return self.calcular_costo_total(horas)

    @cotizar.register
    def _(self, horas: float):                            # cotizar(1.5) -> fracciones de hora
        return self.calcular_costo_total(horas)

    @cotizar.register
    def _(self, opciones: dict):                          # cotizar({"horas": 2, "descuento": 0.1})
        try:
            horas = opciones["horas"]                     # Parámetro obligatorio
        except KeyError as e:
            raise ServicioInvalidoError("Falta el parámetro 'horas' en la cotización.") from e  # Encadenamiento
        return self.calcular_costo_total(horas,
                                         opciones.get("impuesto"),
                                         opciones.get("descuento", 0.0))


class ReservaSala(Servicio):
    """Reserva de salas de reuniones con capacidad máxima."""
    def __init__(self, nombre, tarifa_hora, capacidad, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if not isinstance(capacidad, int) or capacidad <= 0:
            raise ServicioInvalidoError(f"Capacidad '{capacidad}' inválida para la sala.")
        self._capacidad = capacidad

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or not 1 <= horas <= 8:
            raise ReservaInvalidaError(f"Una sala se reserva entre 1 y 8 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        recargo = 1.2 if self._capacidad > 20 else 1.0    # Salas grandes cuestan 20 % más
        return self._tarifa_hora * horas * recargo

    def describir(self):
        return f"Sala {self.id}: {self.nombre} ({self._capacidad} personas) - ${self._tarifa_hora:,.0f}/h"


class AlquilerEquipo(Servicio):
    """Alquiler de equipos (portátiles, video beam, etc.) por unidades."""
    def __init__(self, nombre, tarifa_hora, unidades, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if not isinstance(unidades, int) or unidades <= 0:
            raise ServicioInvalidoError(f"Unidades '{unidades}' inválidas para el equipo.")
        self._unidades = unidades

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or not 1 <= horas <= 72:
            raise ReservaInvalidaError(f"Un equipo se alquila entre 1 y 72 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        seguro = 15000                                    # Valor fijo del seguro del equipo
        return self._tarifa_hora * horas * self._unidades + seguro

    def describir(self):
        return f"Equipo {self.id}: {self.nombre} x{self._unidades} - ${self._tarifa_hora:,.0f}/h"


class Asesoria(Servicio):
    """Asesoría especializada con nivel del asesor."""
    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}   # Multiplicador por nivel

    def __init__(self, nombre, tarifa_hora, nivel, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if nivel not in self.NIVELES:
            raise ServicioInvalidoError(f"Nivel '{nivel}' no existe. Opciones: {list(self.NIVELES)}")
        self._nivel = nivel

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or not 0.5 <= horas <= 4:
            raise ReservaInvalidaError(f"Una asesoría dura entre 0.5 y 4 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        return self._tarifa_hora * horas * self.NIVELES[self._nivel]

    def describir(self):
        return f"Asesoría {self.id}: {self.nombre} (nivel {self._nivel}) - ${self._tarifa_hora:,.0f}/h"


# Prueba rápida de polimorfismo: misma llamada, comportamiento distinto
for s in (ReservaSala("Sala Prueba", 50000, 10),
          AlquilerEquipo("Portátil Prueba", 8000, 2),
          Asesoria("Python Prueba", 90000, "senior")):
    print(f"{s.describir():<60} -> 2 h = ${s.cotizar(2):,.2f}")

Sala SRV-002: Sala Prueba (10 personas) - $50,000/h          -> 2 h = $119,000.00
Equipo SRV-003: Portátil Prueba x2 - $8,000/h                -> 2 h = $55,930.00
Asesoría SRV-004: Python Prueba (nivel senior) - $90,000/h   -> 2 h = $321,300.00


## 5. Clase `Reserva`
Integra **cliente, servicio, duración y estado**. Estados posibles:
`PENDIENTE → CONFIRMADA → PROCESADA`, o `CANCELADA`.
Aquí se usan los bloques **try/except/else**, **try/except/finally** y el **encadenamiento** (`raise ... from ...`).

In [5]:
class Reserva(EntidadBase):
    """Reserva de un servicio por parte de un cliente."""
    PENDIENTE, CONFIRMADA, CANCELADA, PROCESADA = "PENDIENTE", "CONFIRMADA", "CANCELADA", "PROCESADA"

    def __init__(self, cliente, servicio, horas):
        super().__init__("RES")
        if not isinstance(cliente, Cliente):                       # Parámetro faltante o de otro tipo
            raise ReservaInvalidaError("La reserva requiere un cliente válido.")
        if not isinstance(servicio, Servicio):
            raise ReservaInvalidaError("La reserva requiere un servicio válido.")
        if not servicio.disponible:
            raise ServicioNoDisponibleError(f"El servicio '{servicio.nombre}' no está disponible.")
        servicio.validar_parametros(horas)                         # Cada servicio valida a su manera
        self.cliente = cliente
        self.servicio = servicio
        self.horas = horas
        self.__estado = Reserva.PENDIENTE                          # Encapsulado: solo cambia por métodos
        self.costo = None

    @property
    def estado(self):
        return self.__estado

    def confirmar(self, descuento=0.0):
        """Confirma la reserva y calcula su costo (try / except / else)."""
        if self.__estado != Reserva.PENDIENTE:
            raise OperacionNoPermitidaError(f"No se puede confirmar {self.id}: estado {self.__estado}.")
        try:
            costo = self.servicio.calcular_costo_total(self.horas, descuento=descuento)
        except CalculoInconsistenteError as e:
            # Encadenamiento: el error de cálculo queda como causa (__cause__) del error de reserva
            raise ReservaInvalidaError(f"No se pudo confirmar {self.id} por un error de cálculo.") from e
        else:
            # Solo se ejecuta si NO hubo excepción en el try
            self.costo = costo
            self.__estado = Reserva.CONFIRMADA
            logger.info(f"{self.id} CONFIRMADA | {self.cliente.nombre} | {self.servicio.nombre} | ${costo:,.2f}")
            return costo

    def cancelar(self, motivo="Sin motivo"):
        """Cancela la reserva si todavía no fue procesada."""
        if self.__estado in (Reserva.PROCESADA, Reserva.CANCELADA):
            raise OperacionNoPermitidaError(f"No se puede cancelar {self.id}: ya está {self.__estado}.")
        self.__estado = Reserva.CANCELADA
        logger.warning(f"{self.id} CANCELADA | motivo: {motivo}")

    def procesar(self):
        """Procesa (cobra) la reserva confirmada (try / except / finally)."""
        try:
            if self.__estado != Reserva.CONFIRMADA:
                raise OperacionNoPermitidaError(f"Solo se procesan reservas confirmadas ({self.id} está {self.__estado}).")
            self.__estado = Reserva.PROCESADA
            logger.info(f"{self.id} PROCESADA | cobro ${self.costo:,.2f}")
        except OperacionNoPermitidaError:
            logger.error(f"Fallo al procesar {self.id}")
            raise                                                  # Se relanza para que el sistema la gestione
        finally:
            # Se ejecuta SIEMPRE, haya error o no: deja trazabilidad del intento
            logger.debug(f"Intento de procesamiento de {self.id} finalizado (estado: {self.__estado}).")

    def validar(self):
        return self.__estado in (Reserva.PENDIENTE, Reserva.CONFIRMADA, Reserva.CANCELADA, Reserva.PROCESADA)

    def describir(self):
        costo = f"${self.costo:,.2f}" if self.costo else "sin calcular"
        return (f"Reserva {self.id}: {self.cliente.nombre} -> {self.servicio.nombre} "
                f"({self.horas} h) [{self.__estado}] {costo}")

## 6. Clase `SistemaGestion`
Administra las **listas internas** de clientes, servicios y reservas. Cada operación está protegida:
si algo falla, el error se registra en el log y **el programa sigue funcionando**.

In [6]:
class SistemaGestion:
    """Fachada del sistema: guarda los objetos en listas (sin base de datos)."""

    def __init__(self):
        self.clientes = []                        # Lista interna de clientes
        self.servicios = []                       # Lista interna de servicios
        self.reservas = []                        # Lista interna de reservas
        self.exitosas = 0                         # Contadores para el resumen final
        self.fallidas = 0

    def _ejecutar(self, descripcion, funcion, *args, **kwargs):
        """
        Ejecuta cualquier operación de forma segura.
        try/except/else/finally: nunca deja caer la aplicación.
        """
        print(f"\n▶ {descripcion}")
        try:
            resultado = funcion(*args, **kwargs)
        except SoftwareFJError as e:                              # Errores esperados del dominio
            self.fallidas += 1
            causa = f" | causa: {e.__cause__}" if e.__cause__ else ""   # Muestra el encadenamiento
            logger.error(f"{descripcion} -> {type(e).__name__}: {e}{causa}")
            print(f"  ✖ {type(e).__name__}: {e}{causa}")
            return None
        except Exception as e:                                    # Cualquier error inesperado
            self.fallidas += 1
            logger.critical(f"{descripcion} -> ERROR INESPERADO {type(e).__name__}: {e}")
            print(f"  ✖ Error inesperado controlado: {type(e).__name__}: {e}")
            return None
        else:
            self.exitosas += 1
            logger.info(f"{descripcion} -> OK")
            print(f"  ✔ {resultado}")
            return resultado
        finally:
            logger.debug(f"Operación finalizada: {descripcion}")

    # ---- Operaciones públicas ----
    def registrar_cliente(self, nombre, documento, email, telefono):
        def _registrar():
            if any(c.documento == str(documento) for c in self.clientes):   # Evita duplicados
                raise ClienteInvalidoError(f"Ya existe un cliente con documento {documento}.")
            cliente = Cliente(nombre, documento, email, telefono)
            self.clientes.append(cliente)
            return cliente
        return self._ejecutar(f"Registrar cliente '{nombre}'", _registrar)

    def agregar_servicio(self, clase_servicio, *args, **kwargs):
        def _agregar():
            servicio = clase_servicio(*args, **kwargs)
            self.servicios.append(servicio)
            return servicio
        return self._ejecutar(f"Crear servicio {clase_servicio.__name__} {args[:1]}", _agregar)

    def crear_reserva(self, cliente, servicio, horas, descuento=0.0):
        def _reservar():
            reserva = Reserva(cliente, servicio, horas)
            reserva.confirmar(descuento)
            self.reservas.append(reserva)
            return reserva
        nombre = cliente.nombre if isinstance(cliente, Cliente) else cliente
        return self._ejecutar(f"Reservar para {nombre} ({horas} h)", _reservar)

    def cancelar_reserva(self, reserva, motivo):
        def _cancelar():
            if reserva is None:
                raise ReservaInvalidaError("La reserva indicada no existe.")
            reserva.cancelar(motivo)
            return reserva
        return self._ejecutar(f"Cancelar reserva {getattr(reserva, 'id', None)}", _cancelar)

    def procesar_reserva(self, reserva):
        def _procesar():
            if reserva is None:
                raise ReservaInvalidaError("La reserva indicada no existe.")
            reserva.procesar()
            return reserva
        return self._ejecutar(f"Procesar reserva {getattr(reserva, 'id', None)}", _procesar)

    def resumen(self):
        print("\n" + "=" * 70)
        print(f"RESUMEN | Clientes: {len(self.clientes)} | Servicios: {len(self.servicios)} | "
              f"Reservas: {len(self.reservas)}")
        print(f"Operaciones exitosas: {self.exitosas} | Operaciones fallidas controladas: {self.fallidas}")
        print("=" * 70)
        for r in self.reservas:
            print("  •", r)                                       # Polimorfismo: __str__ -> describir()


sistema = SistemaGestion()
print("Sistema listo:", sistema.__class__.__name__)

Sistema listo: SistemaGestion


## 7. Simulación de operaciones
Se ejecutan más de **10 operaciones completas**: registros válidos e inválidos de clientes, creación
correcta e incorrecta de servicios y reservas exitosas y fallidas. El programa **nunca se detiene**.

### 7.1 Registro de clientes

In [7]:
cli_ana   = sistema.registrar_cliente("Ana Pérez", "1012345678", "ana@correo.com", "3101234567")      # 1 válido
cli_luis  = sistema.registrar_cliente("Luis Gómez", "79888777", "luis.gomez@mail.co", "3157654321")   # 2 válido
cli_mal1  = sistema.registrar_cliente("Carl0s", "123456", "carlos@mail.com", "3001112233")            # 3 nombre con número
cli_mal2  = sistema.registrar_cliente("Marta Ruiz", "12AB", "marta@mail.com", "3001112233")           # 4 documento inválido
cli_mal3  = sistema.registrar_cliente("Pedro Díaz", "52000111", "pedro-sin-arroba", "3001112233")     # 5 correo inválido
cli_dup   = sistema.registrar_cliente("Ana Duplicada", "1012345678", "otra@correo.com", "3109998877") # 6 duplicado


▶ Registrar cliente 'Ana Pérez'
  ✔ Cliente CLI-005: Ana Pérez | Doc *******678 | ana@correo.com

▶ Registrar cliente 'Luis Gómez'
  ✔ Cliente CLI-006: Luis Gómez | Doc *****777 | luis.gomez@mail.co

▶ Registrar cliente 'Carl0s'
  ✖ ClienteInvalidoError: [FJ-100] El nombre 'Carl0s' no puede contener números.

▶ Registrar cliente 'Marta Ruiz'
  ✖ ClienteInvalidoError: [FJ-100] Documento '12AB' inválido (6 a 10 dígitos).

▶ Registrar cliente 'Pedro Díaz'
  ✖ ClienteInvalidoError: [FJ-100] Correo 'pedro-sin-arroba' con formato inválido.

▶ Registrar cliente 'Ana Duplicada'
  ✖ ClienteInvalidoError: [FJ-100] Ya existe un cliente con documento 1012345678.


### 7.2 Creación de servicios

In [8]:
sala      = sistema.agregar_servicio(ReservaSala, "Sala Innovación", 60000, capacidad=25)        # 7 válido
portatil  = sistema.agregar_servicio(AlquilerEquipo, "Portátil Dell", 8000, unidades=3)          # 8 válido
asesoria  = sistema.agregar_servicio(Asesoria, "Asesoría en Python", 90000, nivel="senior")      # 9 válido
srv_mal1  = sistema.agregar_servicio(ReservaSala, "Sala Fantasma", -5000, capacidad=10)          # 10 tarifa negativa
srv_mal2  = sistema.agregar_servicio(Asesoria, "Asesoría Redes", 70000, nivel="maestro")         # 11 nivel inexistente
srv_mal3  = sistema.agregar_servicio(AlquilerEquipo, "Video Beam", 12000)                        # 12 falta parámetro 'unidades'


▶ Crear servicio ReservaSala ('Sala Innovación',)
  ✔ Sala SRV-010: Sala Innovación (25 personas) - $60,000/h

▶ Crear servicio AlquilerEquipo ('Portátil Dell',)
  ✔ Equipo SRV-011: Portátil Dell x3 - $8,000/h

▶ Crear servicio Asesoria ('Asesoría en Python',)
  ✔ Asesoría SRV-012: Asesoría en Python (nivel senior) - $90,000/h

▶ Crear servicio ReservaSala ('Sala Fantasma',)
  ✖ ServicioInvalidoError: [FJ-200] Tarifa '-5000' inválida: debe ser un número mayor que 0.

▶ Crear servicio Asesoria ('Asesoría Redes',)
  ✖ ServicioInvalidoError: [FJ-200] Nivel 'maestro' no existe. Opciones: ['junior', 'senior', 'experto']

▶ Crear servicio AlquilerEquipo ('Video Beam',)
  ✖ Error inesperado controlado: TypeError: AlquilerEquipo.__init__() missing 1 required positional argument: 'unidades'


### 7.3 Reservas exitosas y fallidas

In [9]:
res1 = sistema.crear_reserva(cli_ana, sala, 3)                       # 13 exitosa
res2 = sistema.crear_reserva(cli_luis, portatil, 24, descuento=0.10) # 14 exitosa con descuento
res3 = sistema.crear_reserva(cli_ana, asesoria, 2)                   # 15 exitosa
res4 = sistema.crear_reserva(cli_luis, asesoria, 10)                 # 16 duración fuera de rango
res5 = sistema.crear_reserva(cli_mal1, sala, 2)                      # 17 cliente inexistente (None)
res6 = sistema.crear_reserva(cli_luis, sala, 2, descuento=1.5)       # 18 descuento 150 % -> encadenamiento

asesoria.cambiar_disponibilidad(False)                               # Se deshabilita el servicio
res7 = sistema.crear_reserva(cli_luis, asesoria, 1)                  # 19 servicio no disponible
asesoria.cambiar_disponibilidad(True)


▶ Reservar para Ana Pérez (3 h)
  ✔ Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [CONFIRMADA] $257,040.00

▶ Reservar para Luis Gómez (24 h)
  ✔ Reserva RES-016: Luis Gómez -> Portátil Dell (24 h) [CONFIRMADA] $632,961.00

▶ Reservar para Ana Pérez (2 h)
  ✔ Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CONFIRMADA] $321,300.00

▶ Reservar para Luis Gómez (10 h)
  ✖ ReservaInvalidaError: [FJ-300] Una asesoría dura entre 0.5 y 4 horas (se pidió 10).

▶ Reservar para None (2 h)
  ✖ ReservaInvalidaError: [FJ-300] La reserva requiere un cliente válido.

▶ Reservar para Luis Gómez (2 h)
  ✖ ReservaInvalidaError: [FJ-300] No se pudo confirmar RES-020 por un error de cálculo. | causa: [FJ-500] Descuento 150% fuera del rango permitido (0 % a 99 %).

▶ Reservar para Luis Gómez (1 h)
  ✖ ServicioNoDisponibleError: [FJ-210] El servicio 'Asesoría en Python' no está disponible.


### 7.4 Procesamiento, cancelación y operaciones no permitidas

In [10]:
sistema.procesar_reserva(res1)                                       # 20 procesar confirmada
sistema.cancelar_reserva(res1, "El cliente cambió de idea")          # 21 no se puede cancelar algo procesado
sistema.cancelar_reserva(res3, "Asesor incapacitado")                # 22 cancelación válida
sistema.procesar_reserva(res3)                                       # 23 no se procesa una cancelada
sistema.procesar_reserva(res4)                                       # 24 reserva inexistente (None)


▶ Procesar reserva RES-015
  ✔ Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [PROCESADA] $257,040.00

▶ Cancelar reserva RES-015
  ✖ OperacionNoPermitidaError: [FJ-400] No se puede cancelar RES-015: ya está PROCESADA.

▶ Cancelar reserva RES-017
  ✔ Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CANCELADA] $321,300.00

▶ Procesar reserva RES-017
  ✖ OperacionNoPermitidaError: [FJ-400] Solo se procesan reservas confirmadas (RES-017 está CANCELADA).

▶ Procesar reserva None
  ✖ ReservaInvalidaError: [FJ-300] La reserva indicada no existe.


### 7.5 Sobrecarga de métodos (variantes del cálculo de costo)

In [11]:
print("Cálculos para la Sala Innovación (3 h):")
print("  Con IVA (por defecto):        $", f"{sala.calcular_costo_total(3):,.2f}")
print("  Sin impuesto:                 $", f"{sala.calcular_costo_total(3, impuesto=0):,.2f}")
print("  Con IVA y 15 % de descuento:  $", f"{sala.calcular_costo_total(3, descuento=0.15):,.2f}")
print("  cotizar(int 3):               $", f"{sala.cotizar(3):,.2f}")
print("  cotizar(float 2.5):           $", f"{sala.cotizar(2.5):,.2f}")
print("  cotizar(dict):                $", f"{sala.cotizar({'horas': 3, 'descuento': 0.15}):,.2f}")

# Variantes inválidas: se gestionan con el mismo mecanismo seguro del sistema
sistema._ejecutar("Cotizar con texto", sala.cotizar, "tres horas")       # 25 tipo no soportado
sistema._ejecutar("Cotizar sin 'horas'", sala.cotizar, {"descuento": 0.1})  # 26 parámetro faltante (encadenado)

Cálculos para la Sala Innovación (3 h):
  Con IVA (por defecto):        $ 257,040.00
  Sin impuesto:                 $ 216,000.00
  Con IVA y 15 % de descuento:  $ 218,484.00
  cotizar(int 3):               $ 257,040.00
  cotizar(float 2.5):           $ 214,200.00
  cotizar(dict):                $ 218,484.00

▶ Cotizar con texto
  ✖ ServicioInvalidoError: [FJ-200] No se puede cotizar con un argumento de tipo str.

▶ Cotizar sin 'horas'
  ✖ ServicioInvalidoError: [FJ-200] Falta el parámetro 'horas' en la cotización. | causa: 'horas'


### 7.6 Resumen final y contenido del archivo de logs

In [12]:
sistema.resumen()
logger.info("===== Fin de la simulación =====")

# Lectura del archivo de logs con manejo de excepciones
try:
    with open(ARCHIVO_LOG, encoding="utf-8") as archivo:
        lineas = archivo.readlines()
except FileNotFoundError:
    print("No se encontró el archivo de logs.")
else:
    print(f"\nArchivo '{ARCHIVO_LOG}' con {len(lineas)} registros. Errores y advertencias:\n")
    for linea in lineas:
        if any(nivel in linea for nivel in ("ERROR", "WARNING", "CRITICAL")):
            print(linea.rstrip())
finally:
    print("\nFin de la ejecución: la aplicación se mantuvo estable en todo momento.")


RESUMEN | Clientes: 2 | Servicios: 3 | Reservas: 3
Operaciones exitosas: 10 | Operaciones fallidas controladas: 16
  • Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [PROCESADA] $257,040.00
  • Reserva RES-016: Luis Gómez -> Portátil Dell (24 h) [CONFIRMADA] $632,961.00
  • Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CANCELADA] $321,300.00

Archivo 'software_fj.log' con 64 registros. Errores y advertencias:

2026-10-28 13:20:25 | ERROR    | Registrar cliente 'Carl0s' -> ClienteInvalidoError: [FJ-100] El nombre 'Carl0s' no puede contener números.
2026-10-28 13:20:25 | ERROR    | Registrar cliente 'Marta Ruiz' -> ClienteInvalidoError: [FJ-100] Documento '12AB' inválido (6 a 10 dígitos).
2026-10-28 13:20:25 | ERROR    | Registrar cliente 'Pedro Díaz' -> ClienteInvalidoError: [FJ-100] Correo 'pedro-sin-arroba' con formato inválido.
2026-10-28 13:20:25 | ERROR    | Registrar cliente 'Ana Duplicada' -> ClienteInvalidoError: [FJ-100] Ya existe un cliente con documento 101234

## Conclusión técnica
- **Abstracción:** `EntidadBase` y `Servicio` definen contratos que las hijas deben cumplir.
- **Herencia y polimorfismo:** `ReservaSala`, `AlquilerEquipo` y `Asesoria` sobrescriben `calcular_costo`, `describir` y `validar_parametros`.
- **Encapsulación:** datos del cliente y estado de la reserva son privados y se modifican solo con validación.
- **Sobrecarga:** parámetros opcionales y `singledispatchmethod`.
- **Excepciones:** jerarquía propia, `try/except`, `try/except/else`, `try/except/finally`, `raise ... from` y registro de todo en `software_fj.log`.